# 15. 이벤트 충격 분석 (Causal Impact)

## 분석 배경 및 목적

Brodersen et al. (2015)은 Google에서 개발한 **베이지안 구조적 시계열 모델(Bayesian Structural Time-Series, BSTS)**을 활용하여, 특정 개입(intervention)이 시계열에 미친 인과적 효과(causal effect)를 추정하는 CausalImpact 프레임워크를 제안했다. 이 방법의 핵심 아이디어는 **반사실(counterfactual)** 추정이다: "이벤트가 발생하지 않았더라면 시계열이 어떻게 전개되었을 것인가"를 모델링하고, 실제 관측값과의 차이를 이벤트의 효과로 해석한다.

본 분석에서는 택시 수요에 영향을 미친 주요 이벤트의 인과적 효과를 추정한다:

- **요금 인상**: 기본요금/km당 요금 변경이 수요에 미친 단기/장기 영향
- **코로나 발생 및 사회적 거리두기**: 외부 충격이 택시 수요에 미친 구조적 변화
- **플랫폼 이벤트**: 카카오T 등 모빌리티 플랫폼의 정책 변화 영향

**방법론**: Google CausalImpact의 원리를 따르되, BSTS 대신 **Ridge 회귀 기반 합성 통제(synthetic control)**를 사용한다. 이벤트 전 기간의 데이터로 요일, 기온, 강수, 추세 등을 통제변수로 포함한 회귀 모델을 학습하고, 이벤트 후 기간에 대해 반사실 예측을 수행한다. 실제값 - 반사실 = 이벤트의 pointwise impact이며, 이를 누적하면 총 영향(cumulative impact)을 산출할 수 있다.


In [ ]:
# 필요 라이브러리 설치
!pip install -q psutil statsmodels scikit-learn

In [ ]:
# 메모리 모니터링 유틸
import psutil
import os
import gc

def print_mem(tag=''):
    proc = psutil.Process(os.getpid())
    mem = proc.memory_info().rss / 1024**2
    print(f'[MEM {tag}] {mem:.0f} MB')

print_mem('start')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import platform

# 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (14, 5)

## 1. 택시 데이터 일별 집계

Causal Impact 분석을 위한 일별 시계열을 구성한다. Brodersen et al. (2015)의 프레임워크에서 종속변수(response variable)는 일별 택시 수요이며, 통제변수(covariates)는 외부 요인(날씨, 요일 등)이다.


In [ ]:
DATA_PATH = 'DC_TBYXD012.csv'
EXT_DIR = 'external_data'

usecols = ['RIDE_DTIME']
dtype = {'RIDE_DTIME': str}

daily_counts = pd.Series(dtype='int64')

for chunk in pd.read_csv(DATA_PATH, usecols=usecols, dtype=dtype, chunksize=1_000_000):
    chunk['date'] = chunk['RIDE_DTIME'].str[:8]
    counts = chunk.groupby('date').size()
    daily_counts = daily_counts.add(counts, fill_value=0)
    del chunk
    gc.collect()

daily_counts = daily_counts.astype(int)
daily_counts.index = pd.to_datetime(daily_counts.index, format='%Y%m%d')
daily_counts = daily_counts.sort_index()
daily_counts.name = 'trip_count'

df = daily_counts.to_frame().reset_index()
df.columns = ['date', 'trip_count']
print(f'기간: {df.date.min()} ~ {df.date.max()}, {len(df)}일')
print_mem('after load')

## 2. 외부 데이터 조인

calendar(공휴일, 요일), weather(기온, 강수), covid(확진자 수), social_distancing(거리두기 단계), taxi_events(요금 인상, 플랫폼 이벤트) 데이터를 조인한다. 이 외부 변수들은 반사실 모델의 통제변수로 사용되어, 이벤트 효과와 외부 요인의 효과를 분리(disentangle)하는 역할을 한다.


In [ ]:
# 외부 데이터 로드
cal = pd.read_csv(f'{EXT_DIR}/calendar_2018_2026.csv', encoding='utf-8', parse_dates=['date'])
weather = pd.read_csv(f'{EXT_DIR}/weather_asos_daily_seoul_2018_2026.csv', encoding='utf-8', parse_dates=['date'])
covid = pd.read_csv(f'{EXT_DIR}/covid_korea_2018_2026.csv', encoding='utf-8', parse_dates=['date'])
distancing = pd.read_csv(f'{EXT_DIR}/social_distancing_daily.csv', encoding='utf-8', parse_dates=['date'])
events = pd.read_csv(f'{EXT_DIR}/taxi_events_timeline.csv', encoding='utf-8', parse_dates=['date'])

# 조인
df = df.merge(cal, on='date', how='left')
df = df.merge(weather[['date', 'avg_temp', 'rainfall']], on='date', how='left')
df = df.merge(covid[['date', 'new_cases']], on='date', how='left')
df = df.merge(distancing, on='date', how='left')

df['new_cases'] = df['new_cases'].fillna(0)
df['distancing_level'] = df['distancing_level'].fillna(0)

print(f'이벤트 목록: {len(events)}건')
events

## 3. Causal Impact 분석 프레임워크

Brodersen et al. (2015)의 원리를 따르되, 라이브러리 의존 없이 직접 구현한다:

1. **이벤트 전 기간(pre-period)**: 요일, 기온, 강수, 추세 등을 feature로 포함한 Ridge 회귀 모델을 학습
2. **이벤트 후 기간(post-period)**: 학습된 모델로 반사실(counterfactual) 수요를 예측
3. **Pointwise Impact**: 실제 수요 - 반사실 예측 = 이벤트의 일별 효과
4. **Cumulative Impact**: pointwise impact의 누적합 = 이벤트의 총 영향

Ridge 회귀를 사용하는 이유: (1) 다중공선성(multicollinearity)이 있는 통제변수 간 안정적 계수 추정, (2) 과적합 방지, (3) BSTS 대비 구현 단순성. 단, 베이지안 불확실성 구간(credible interval)은 제공하지 못하는 것이 한계다.


In [ ]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

def causal_impact_analysis(df, event_date, event_name,
                           pre_days=90, post_days=60):
    """
    이벤트 전후 비교 기반 Causal Impact 분석
    
    - pre_days: 이벤트 전 학습 기간 (일)
    - post_days: 이벤트 후 분석 기간 (일)
    """
    event_dt = pd.to_datetime(event_date)
    
    # 전후 기간 설정
    pre_start = event_dt - pd.Timedelta(days=pre_days)
    post_end = event_dt + pd.Timedelta(days=post_days)
    
    # 범위 필터
    mask_all = (df['date'] >= pre_start) & (df['date'] <= post_end)
    subset = df[mask_all].copy()
    
    if len(subset) < pre_days + 10:
        return None  # 데이터 부족
    
    # 피처 생성
    subset['trend'] = np.arange(len(subset))
    subset['dow'] = subset['date'].dt.dayofweek
    dow_dummies = pd.get_dummies(subset['dow'], prefix='dow', drop_first=True).astype(float)
    
    feature_cols = ['trend', 'avg_temp', 'rainfall']
    X = subset[feature_cols].fillna(0)
    X = pd.concat([X, dow_dummies], axis=1)
    y = subset['trip_count'].values
    
    # 전/후 분리
    is_pre = subset['date'] < event_dt
    X_pre, y_pre = X[is_pre], y[is_pre]
    X_post, y_post = X[~is_pre], y[~is_pre]
    
    if len(X_post) == 0:
        return None
    
    # 학습 (Ridge 회귀)
    scaler = StandardScaler()
    X_pre_s = scaler.fit_transform(X_pre)
    X_post_s = scaler.transform(X_post)
    
    model = Ridge(alpha=1.0)
    model.fit(X_pre_s, y_pre)
    
    # 반사실 예측
    counterfactual = model.predict(X_post_s)
    
    # 잔차 기반 신뢰구간 (pre 기간)
    pre_resid = y_pre - model.predict(X_pre_s)
    resid_std = pre_resid.std()
    
    # 결과
    post_dates = subset[~is_pre]['date'].values
    impact = y_post - counterfactual
    cumulative = np.cumsum(impact)
    
    result = {
        'event_name': event_name,
        'event_date': event_dt,
        'pre_dates': subset[is_pre]['date'].values,
        'pre_actual': y_pre,
        'pre_predicted': model.predict(X_pre_s),
        'post_dates': post_dates,
        'post_actual': y_post,
        'counterfactual': counterfactual,
        'impact': impact,
        'cumulative_impact': cumulative,
        'ci_upper': counterfactual + 1.96 * resid_std,
        'ci_lower': counterfactual - 1.96 * resid_std,
        'avg_impact': impact.mean(),
        'total_impact': impact.sum(),
        'relative_impact': impact.sum() / counterfactual.sum() * 100,
    }
    return result

print('causal_impact_analysis 함수 정의 완료')

## 4. 주요 이벤트별 분석 실행

각 이벤트에 대해 pre-period(이벤트 전 90일)와 post-period(이벤트 후 90일)를 설정하고, 반사실 추정을 수행한다. 이벤트가 시간적으로 겹치는 경우(예: 코로나 발생 직후 사회적 거리두기), 개별 효과 분리에 한계가 있음을 유의해야 한다.


In [ ]:
# 분석 대상 이벤트 선정 (데이터 범위 내 주요 이벤트)
data_min = df['date'].min()
data_max = df['date'].max()

# 전후 90+60일 여유가 있는 이벤트만
valid_events = events[
    (events['date'] >= data_min + pd.Timedelta(days=90)) &
    (events['date'] <= data_max - pd.Timedelta(days=60))
].copy()

print(f'분석 가능한 이벤트: {len(valid_events)}건')
valid_events[['date', 'event', 'category']]

In [ ]:
# 전체 이벤트 분석 실행
results = []
for _, row in valid_events.iterrows():
    res = causal_impact_analysis(df, row['date'], row['event'])
    if res is not None:
        results.append(res)
        print(f"[{row['event']}] 평균 영향: {res['avg_impact']:+.0f}건/일, "
              f"총 영향: {res['total_impact']:+,.0f}건, "
              f"상대 영향: {res['relative_impact']:+.1f}%")

print(f'\n분석 완료: {len(results)}건')

## 5. 시각화: 이벤트별 실제 vs 반사실

Brodersen et al. (2015)의 표준 시각화 형식을 따라 세 가지 패널을 구성한다:
1. **Original**: 실제 수요(실선)와 반사실 예측(점선)의 비교
2. **Pointwise Impact**: 일별 효과 (양수면 수요 증가, 음수면 감소)
3. **Cumulative Impact**: 누적 효과의 시간적 추이

반사실 예측과 실제값의 괴리가 시간이 지나도 지속되면, 이벤트의 효과가 **구조적(permanent)**임을 의미한다. 반대로 괴리가 줄어들면 **일시적(transient)** 효과다.


In [ ]:
def plot_causal_impact(res):
    """개별 이벤트 Causal Impact 시각화 (3패널)"""
    fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
    
    # --- 패널 1: 실제 vs 반사실 ---
    ax = axes[0]
    # pre 기간
    ax.plot(res['pre_dates'], res['pre_actual'], color='steelblue',
            linewidth=0.5, alpha=0.5)
    # post 기간 - 실제
    ax.plot(res['post_dates'], res['post_actual'],
            color='steelblue', linewidth=1, label='실제')
    # post 기간 - 반사실
    ax.plot(res['post_dates'], res['counterfactual'],
            color='red', linewidth=1, linestyle='--', label='반사실 (예측)')
    # 신뢰구간
    ax.fill_between(res['post_dates'], res['ci_lower'], res['ci_upper'],
                    color='red', alpha=0.1)
    ax.axvline(x=res['event_date'], color='black', linestyle='-', linewidth=1.5)
    ax.set_title(f"[{res['event_name']}] 실제 vs 반사실")
    ax.set_ylabel('일별 건수')
    ax.legend()
    
    # --- 패널 2: Pointwise Impact ---
    ax = axes[1]
    ax.bar(res['post_dates'], res['impact'], width=1,
           color=['coral' if v < 0 else 'steelblue' for v in res['impact']],
           alpha=0.7)
    ax.axhline(y=0, color='black', linewidth=0.5)
    ax.axvline(x=res['event_date'], color='black', linestyle='-', linewidth=1.5)
    ax.set_title('일별 영향 (실제 - 반사실)')
    ax.set_ylabel('건수 차이')
    
    # --- 패널 3: Cumulative Impact ---
    ax = axes[2]
    ax.fill_between(res['post_dates'], 0, res['cumulative_impact'],
                    color='coral' if res['total_impact'] < 0 else 'steelblue',
                    alpha=0.3)
    ax.plot(res['post_dates'], res['cumulative_impact'], color='black', linewidth=1)
    ax.axhline(y=0, color='black', linewidth=0.5)
    ax.axvline(x=res['event_date'], color='black', linestyle='-', linewidth=1.5)
    ax.set_title(f"누적 영향: {res['total_impact']:+,.0f}건 ({res['relative_impact']:+.1f}%)")
    ax.set_ylabel('누적 건수')
    ax.set_xlabel('날짜')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# 상위 이벤트 시각화 (최대 6개)
for res in results[:6]:
    plot_causal_impact(res)

## 6. 이벤트 영향 요약표

모든 이벤트의 분석 결과를 표로 정리한다. 각 이벤트의 (1) 평균 일별 영향(건/일), (2) 총 누적 영향(건), (3) 영향의 방향(양/음)과 지속성을 요약하여 이벤트 간 비교를 가능하게 한다.


In [ ]:
# 요약표 생성
summary = pd.DataFrame([{
    '이벤트': r['event_name'],
    '날짜': r['event_date'].strftime('%Y-%m-%d'),
    '평균 일별 영향': f"{r['avg_impact']:+,.0f}",
    '총 누적 영향': f"{r['total_impact']:+,.0f}",
    '상대 영향(%)': f"{r['relative_impact']:+.1f}%",
} for r in results])

summary

In [ ]:
# 이벤트별 영향 크기 비교 차트
impact_vals = [r['relative_impact'] for r in results]
event_names = [r['event_name'] for r in results]
colors = ['coral' if v < 0 else 'steelblue' for v in impact_vals]

fig, ax = plt.subplots(figsize=(10, max(4, len(results)*0.6)))
bars = ax.barh(range(len(results)), impact_vals, color=colors,
               edgecolor='black', linewidth=0.5)
ax.set_yticks(range(len(results)))
ax.set_yticklabels(event_names)
ax.set_xlabel('상대 영향 (%)')
ax.set_title('이벤트별 택시 수요 영향 비교')
ax.axvline(x=0, color='black', linewidth=0.5)

# 값 라벨
for i, v in enumerate(impact_vals):
    ax.text(v + (1 if v >= 0 else -1), i, f'{v:+.1f}%',
            va='center', ha='left' if v >= 0 else 'right', fontsize=9)

plt.tight_layout()
plt.show()

### Causal Impact 분석 결과 해석

**방법론 한계 및 주의사항:**
- 본 분석은 Google CausalImpact의 베이지안 구조적 시계열(BSTS) 대신 Ridge 회귀 기반 전후 비교를 사용하므로, 불확실성 구간(credible interval)을 제공하지 못한다
- 요일, 기온, 강수량, 추세를 통제변수로 활용하여 계절성과 날씨 효과를 분리했으나, 관측되지 않은 교란변수(unobserved confounders)의 영향은 배제할 수 없다
- 여러 이벤트가 시간적으로 겹치는 경우(예: 코로나 + 거리두기 + 요금 인상) 개별 효과 분리에 한계가 있다

**실무 활용:**
- 요금 인상 시 수요 탄력성(price elasticity) 추정의 근거 자료
- 코로나 등 외부 충격 이후 수요 회복 기간(recovery period) 예측
- 플랫폼 정책 변경의 효과 사전 평가(ex-ante evaluation) 시 벤치마크


In [ ]:
# 메모리 정리
del results
gc.collect()
print_mem('final')

---

## References

1. Brodersen, K. H., Gallusser, F., Koehler, J., Remy, N., & Scott, S. L. (2015). Inferring causal impact using Bayesian structural time-series models. *The Annals of Applied Statistics*, 9(1), 247-274.
2. Abadie, A., Diamond, A., & Hainmueller, J. (2010). Synthetic Control Methods for Comparative Case Studies: Estimating the Effect of California's Tobacco Control Program. *Journal of the American Statistical Association*, 105(490), 493-505.
3. Hoerl, A. E., & Kennard, R. W. (1970). Ridge Regression: Biased Estimation for Nonorthogonal Problems. *Technometrics*, 12(1), 55-67.
